# 2장 연습 — 텍스트 데이터 다루기 (내 손으로)

**규칙**
1. 책을 보면서 **직접 쳐서** 채운다. `ch02.ipynb` 에서 복사·붙여넣기 금지.
2. 🔮 표시가 있는 셀은 **돌리기 전에** 결과를 `# 예측:` 줄에 먼저 적는다. 틀리면 그 자리가 공부할 곳이다.
3. 🔨 는 일부러 깨 보는 칸이다. 값을 바꿔 보고 무엇이 왜 달라졌는지 한 줄 적는다.
4. ✅ 확인 셀은 **고치지 말고 돌리기만** 한다. 통과하면 `OK`, 아니면 무엇이 틀렸는지 알려 준다.
5. 맨 끝 🏁 는 책을 덮고 빈 칸에서 다시 짠다. 막히면 그 부분만 보고 다시 덮는다.

막히면 셀 내용이나 에러를 그대로 보내 주세요.

In [ ]:
# 책 2장 데이터. 다운로드 대신 저장소에 있는 파일을 쓴다.
DATA = "../../ch02/01_main-chapter-code/the-verdict.txt"

## 2.2 텍스트 토큰화하기
`raw_text` 로 파일을 읽고 글자 수와 처음 99글자를 출력한다.

🔮 `re.split(r'(\s)', "Hello, world. This, is a test.")` 의 결과를 먼저 적고 돌린다.

In [ ]:
# 예측:

쉼표·마침표·공백을 나누고, 빈 문자열과 공백을 지운다. 그다음 `--`, `?`, `!` 등까지 나누는 정규식으로 `raw_text` 전체를 쪼개 `preprocessed` 에 담는다.

🔨 정규식에서 `--` 를 빼면 토큰 수가 늘까 줄까? 예측하고 확인한다.

In [ ]:
# 예측:

✅ 확인

In [ ]:
assert len(preprocessed) == 4690, f"토큰 수가 {len(preprocessed)} — 책은 4690. 정규식이나 공백 지우기를 다시 보자"
assert "" not in preprocessed and " " not in preprocessed, "빈 문자열이나 공백이 남아 있다"
print("OK 2.2")

## 2.3 토큰을 토큰 ID로 변환하기
`vocab` (토큰 → 번호) 을 만들고 `SimpleTokenizerV1` (encode·decode) 을 짠다.

🔮 `vocab` 의 크기는? 왜 4690 보다 작은가?

In [ ]:
# 예측:

✅ 확인

In [ ]:
assert len(vocab) == 1130, f"단어장 크기 {len(vocab)} — 책은 1130"
_t = SimpleTokenizerV1(vocab)
_s = '"It\'s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'
_ids = _t.encode(_s)
assert all(isinstance(i, int) for i in _ids), "encode 는 정수 번호 목록을 내야 한다"
assert _t.encode(_t.decode(_ids)) == _ids, "decode 한 글을 다시 encode 하면 같은 번호가 나와야 한다"
print("OK 2.3")

🔨 단어장에 없는 단어(`"Hello"`)를 encode 하면 무슨 일이 생기나? 예측하고 돌려 본다.

In [ ]:
# 예측:

## 2.4 특수 문맥 토큰 추가하기
`<|endoftext|>`, `<|unk|>` 를 단어장에 넣고, 모르는 단어를 `<|unk|>` 로 바꾸는 `SimpleTokenizerV2` 를 짠다.

✅ 확인

In [ ]:
assert len(vocab) == 1132, f"단어장 크기 {len(vocab)} — 특수 토큰 둘을 더해 1132"
_t = SimpleTokenizerV2(vocab)
_ids = _t.encode("Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.")
assert _ids[0] == vocab["<|unk|>"], "Hello 는 <|unk|> 가 되어야 한다"
assert vocab["<|endoftext|>"] in _ids, "<|endoftext|> 가 번호로 바뀌어야 한다"
print("OK 2.4")

## 2.5 바이트 페어 인코딩 (BPE)
`tiktoken` 의 `gpt2` 인코딩으로 아래 문장을 encode·decode 한다.

In [ ]:
text = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace."

🔮 `someunknownPlace` 는 단어장에 없는데 BPE 는 에러를 낼까? `<|unk|>` 를 쓸까? 둘 다 아니면?

In [ ]:
# 예측:

✅ 확인 (encode 결과를 `integers` 에 담았다고 본다)

In [ ]:
assert integers == [15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13], "allowed_special 를 확인하자"
print("OK 2.5")

## 2.6 슬라이딩 윈도로 데이터 샘플링하기
입력–타깃 쌍을 눈으로 확인한 뒤, `GPTDatasetV1` 과 `create_dataloader_v1(txt, batch_size, max_length, stride, shuffle, drop_last, num_workers)` 를 짠다.

🔮 `stride` 를 `max_length` 보다 작게 하면 무엇이 달라지나? 장점과 단점은?

In [ ]:
# 예측:

✅ 확인

In [ ]:
import torch
with open(DATA, "r", encoding="utf-8") as f:
    _raw = f.read()
_dl = create_dataloader_v1(_raw, batch_size=8, max_length=4, stride=4, shuffle=False)
_x, _y = next(iter(_dl))
assert tuple(_x.shape) == (8, 4) and tuple(_y.shape) == (8, 4), f"모양이 {tuple(_x.shape)} — (8, 4) 여야 한다"
assert _x[0].tolist() == [40, 367, 2885, 1464], f"첫 입력 {_x[0].tolist()}"
assert torch.equal(_x[:, 1:], _y[:, :-1]), "타깃은 입력을 한 칸 민 것이어야 한다"
print("OK 2.6")

## 2.7 토큰 임베딩 만들기
`torch.nn.Embedding(6, 3)` 을 만들고 `input_ids = torch.tensor([2, 3, 5, 1])` 을 넣어 본다.

🔮 `embedding_layer(torch.tensor([3]))` 은 `embedding_layer.weight` 의 어느 부분과 같을까?

In [ ]:
# 예측:

## 2.8 단어 위치 인코딩하기
어휘 50257, 출력 차원 256 의 토큰 임베딩 + 문맥 길이 4 의 위치 임베딩을 더해 `input_embeddings` 를 만든다 (배치 8, 길이 4).

✅ 확인

In [ ]:
assert tuple(input_embeddings.shape) == (8, 4, 256), f"모양이 {tuple(input_embeddings.shape)} — (8, 4, 256)"
print("OK 2.8")

## 🏁 책 덮고 다시 짜기

빈 칸에 `MyTokenizer` 를 처음부터 짠다: 글을 쪼갠다 → 단어장을 만든다 → 번호를 붙인다 → encode·decode, 모르는 단어는 `<|unk|>`.
이름·순서는 책과 달라도 된다. 아래 확인 셀만 통과하면 된다.

In [ ]:
with open(DATA, "r", encoding="utf-8") as f:
    _raw = f.read()
_t = MyTokenizer(_raw)          # 원문 글을 받아 단어장을 스스로 만든다
_s = "I HAD always thought Jack Gisburn rather a cheap genius"
_ids = _t.encode(_s)
assert _t.decode(_ids) == _s, "되돌리기 실패 — 쉼표·따옴표가 없는 문장은 원문 그대로 돌아와야 한다"
assert _t.encode(_t.decode(_ids)) == _ids
assert _t.decode(_t.encode("Hello Jack")) == "<|unk|> Jack", "모르는 단어 처리"
print("OK 🏁 — PROGRESS.md 에 2장 완료를 적고 커밋·푸시!")